# NBA Predictive Analytics: Machine Learning Accolade Forecasting & Probabilistic Matchup Engine
**Author:** Raul Diaz  
**Project Overview:** This end-to-end data science pipeline takes raw player performance statistics, applies supervised machine learning to predict postseason accolade probabilities, and aggregates player-level metrics into an algorithmic matchup simulation engine to isolate wagering inefficiencies.

---
### Technical Architecture & Workflow
1. **Data Ingestion & Preprocessing:** Web-scraping structural data, handling repeating web headers, and casting schema types.
2. **Feature Engineering & Target Labeling:** Transforming categorical award metrics into distinct binary classification labels.
3. **Supervised Learning (Random Forest Classifier):** Training a predictive model optimized for high-precision output to handle severe class imbalance.
4. **Feature Importance & Insights Visualization:** Extracting and plotting predictive feature dependencies.
5. **Algorithmic Team Aggregation & Matchup Simulation:** Transforming atomic player metrics into a composite team-strength matrix to run probabilistic game simulations.

### 1. Data Ingestion & Data Frame Initialization

In [ ]:
import pandas as pd

url = "https://www.basketball-reference.com/leagues/NBA_2025_per_game.html"
tables = pd.read_html(url)
df = tables[0]

df = df[df["Player"] != 'Player'].copy()

In [ ]:
df.head()

In [ ]:
df.columns

In [ ]:
df = df.drop_duplicates(subset='Player', keep='first')

In [ ]:
df.head()

In [ ]:
df.columns

In [ ]:
df.isna().sum()

In [ ]:
cols_to_fix = ['FG%', '3P%', '2P%', 'eFG%', 'FT%']
df[cols_to_fix] = df[cols_to_fix].fillna(0)

In [ ]:
# all rows in cols_to_fix
df.loc[:, cols_to_fix] = df.loc[:, cols_to_fix].fillna(0)

In [ ]:
df.head()

In [ ]:
df.columns

#### Clean the Awards column without overwriting the entire DataFrame

In [ ]:
df[['Awards']] = df[['Awards']].fillna('None')

#### Clean up rows where the header repeats in the data, then drop missing records

In [ ]:
df = df[df['Rk'] != 'Rk']
df = df.dropna(subset=['Rk'])

#### Convert core stats from strings to floats/ints 

In [ ]:
numeric_cols = ['Age', 'G', 'GS', 'MP', 'FG', 'FGA', '3P', '3PA', '2P', '2PA', 
                'FT', 'FTA', 'ORB', 'DRB', 'TRB', 'AST', 'STL', 'BLK', 'TOV', 'PF', 'PTS']

for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors='coerce')

df[numeric_cols] = df[numeric_cols].fillna(0)

df.head()

#### create a column that tells the machine learning model whether a player won an award (1) or didn't (0).

In [ ]:
df['Won_Award'] = df['Awards'].apply(lambda x: 0 if x == 'None' or x == '' else 1)
print(df['Won_Award'].value_counts())

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score

####  Define the features (stats) the model will look at

In [ ]:
features = ['MP', 'PTS', 'TRB', 'AST', 'STL', 'BLK', 'FG%', '3P%', 'FT%', 'TOV']

In [ ]:
X = df[features]
y = df['Won_Award']

#### Split the data: 80% to train the model, 20% to test its accuracy

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

#### Initialize and train the Random Forest

In [ ]:
model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

#### Test the model

In [ ]:
y_pred = model.predict(X_test)
print(f"Model Accuracy: {accuracy_score(y_test, y_pred) * 100:.2f}%")
print("\nClassification Report:\n", classification_report(y_test, y_pred))

* **Class 0 (No Award):** Players who did not receive any award recognition.
* **Class 1 (Won Award):** Players who made an All-Star, All-NBA team, or won a major award.

#### 1. Precision (Predictive Reliability)
$$\text{Precision} = \frac{\text{True Positives}}{\text{True Positives} + \text{False Positives}}$$
* **Class 1 Precision (0.75):** When our AI predicts that a player *will* win an award, it is correct **75% of the time**. In sports betting terms, if the model flags a player as a "future award winner," you can trust that prediction with high confidence.

#### 2. Recall (Sensitivity / Detection Rate)
$$\text{Recall} = \frac{\text{True Positives}}{\text{True Positives} + \text{False Negatives}}$$
* **Class 1 Recall (0.27):** The model only successfully identifies **27%** of the actual award winners in the dataset. Because award winners are so rare, the model is being highly conservative and missing a lot of players who ended up winning awards (low sensitivity).

#### 3. F1-Score (The Balance Metric)
The F1-score is the harmonic mean of Precision and Recall. 
* Our Class 1 F1-score is **0.40**, reflecting that while our model is quite accurate when it *does* make an award prediction (high precision), it misses too many actual winners (low recall).

#### 4. The Accuracy Paradox (0.92)
At first glance, an overall accuracy of **92%** looks amazing. However, because $90\%$ of the players in our test sample are non-winners (103 out of 114), a dumb model that blindly guesses "No Award" for every single player would automatically achieve $90\%$ accuracy. 

### 💡 Key Takeaway for Portfolio & Gambling Models:
Our model acts like a **strict filter**. It doesn't hand out award predictions easily (low recall), but when it *does* say a player will win an award, it has a 75% accuracy rate (high precision). 

To improve this model for future betting/predictive value, we should explore **SMOTE (Synthetic Minority Over-sampling Technique)** or adjust the class weights to help the model better recognize the unique statistical signatures of award-winning players.

#### Find Out Which Stats Matter Most

In [ ]:
# Get feature importances from the model
importances = pd.DataFrame({
    'Stat': features,
    'Importance': model.feature_importances_
}).sort_values(by='Importance', ascending=False)

print("Which stats matter most to the AI model?")
print(importances)

#### Generate the Predictions

In [ ]:
df.columns

In [ ]:
df['Award_Probability'] = model.predict_proba(X)[:, 1]

# top 15 players most likely to win an award 
top_predictions = df[['Player', 'Team', 'PTS', 'TRB', 'AST', 'Award_Probability', 'Awards']].sort_values(by='Award_Probability', ascending=False)
top_predictions.head(15)

#### Which Stats Matter Most (to the Random Forest Model)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Calculate feature importance from the trained Random Forest model
importances = model.feature_importances_
feature_imp_df = pd.DataFrame({"Stat": features, 'Importance': importances}).sort_values(by='Importance', ascending=False)

# Visualizing predictive dependencies
plt.figure(figsize=(10, 5))
sns.barplot(x='Importance', y='Stat', data=feature_imp_df, palette='viridis')
plt.title('Predictive Feature Importance: Award Recognition Model')
plt.xlabel('Statistical Weight (Importance Score)')
plt.ylabel('NBA Metric')
plt.show()

# Part 2

### Code block to aggregate player award probabilities into team-level scores and build the matchup game

In [ ]:
# Filter for main rotation players ( at least 10 minutes in game )
rotation_players = df[df["MP"] >= 10].copy()

# Sort by team and minutes played, then isolate the top 8 players per team
top_8_players = rotation_players.sort_values(['Team', 'MP'], ascending=[True, False]).groupby('Team').head(8)

In [ ]:
top_8_players

In [ ]:
# Aggregate individual statistics and ML award probabilities into a team profile
team_profiles = top_8_players.groupby('Team').agg({
    'PTS': 'sum',                # Total scoring power of the rotation
    'TRB': 'sum',                # Total rebounding presence
    'AST': 'sum',                # Playmaking and ball movement
    'STL': 'sum',                # Defensive disruption
    'BLK': 'sum',                # Rim protection
    'TOV': 'sum',                # Turnovers (lower is better)
    'Award_Probability': 'sum'   # Concentration of elite/star talent on the roster
}).reset_index()

# Rename columns to make the dataset clean and descriptive
team_profiles.columns = ['Team', 'Core_PTS', 'Core_TRB', 'Core_AST', 'Core_STL', 'Core_BLK', 'Core_TOV', 'Star_Power']

print("Team Profiles Successfully Generated!")
team_profiles.sort_values(by='Star_Power', ascending=False).head()

### Matchup Probability

In [ ]:
import numpy as np

def simulate_matchup(team_a, team_b, data):
    """
    Simulates a matchup between Team A and Team B based on aggregated rotation data,
    incorporating individual player metrics and ML award probabilities.
    """
    try:
        stats_a = data[data['Team'] == team_a].iloc[0]
        stats_b = data[data['Team'] == team_b].iloc[0]
    except IndexError:
        return f"Error: One of the teams ('{team_a}' or '{team_b}') was not found. Check your team abbreviations!"
    
    # Calculate performance differentials (Team A vs Team B)
    pts_diff  = stats_a['Core_PTS'] - stats_b['Core_PTS']
    def_diff  = (stats_a['Core_STL'] + stats_a['Core_BLK']) - (stats_b['Core_STL'] + stats_b['Core_BLK'])
    tov_diff  = stats_a['Core_TOV'] - stats_b['Core_TOV'] # Negative means fewer turnovers (advantage)
    star_diff = stats_a['Star_Power'] - stats_b['Star_Power'] # Elite talent gap
    
    # Linear logit score combining team metrics and machine learning features
    # Higher score = greater structural advantage for Team A
    matchup_score = (pts_diff * 0.3) + (def_diff * 0.4) - (tov_diff * 0.3) + (star_diff * 2.5)
    
    # Pass through a sigmoid function to map the score into a real win probability (0 to 1)
    prob_a = 1 / (1 + np.exp(-matchup_score * 0.1))
    prob_b = 1 - prob_a
    
    winner = team_a if prob_a > prob_b else team_b
    winning_percentage = max(prob_a, prob_b) * 100
    
    # Calculate Vegas implied moneyline odds from the model's probability
    if winning_percentage > 50:
        implied_odds = -100 / ((winning_percentage / 100) - 1)
        odds_str = f"-{abs(implied_odds):.0f}"
    else:
        implied_odds = 100 / (winning_percentage / 100)
        odds_str = f"+{implied_odds:.0f}"
        
    print(f"Matchup Simulation: {team_a} vs {team_b}")
    print(f"Predicted Winner: {winner}")
    print(f"Model Win Probability: {winning_percentage:.1f}%")
    print(f"Equivalent Fair Market Odds: {odds_str}\n")
    
    return prob_a

In [ ]:
simulate_matchup('OKC', 'IND', team_profiles)

In [ ]:
simulate_matchup('IND', 'NYK', team_profiles)

In [ ]:
simulate_matchup('BOS', 'LAL', team_profiles)

### Project Summary & Backtesting Analysis

To evaluate the predictive power of this end-to-end analytics pipeline, the matchup simulation engine was backtested against key high-stakes series from the 2025 postseason:

1. **The 2025 NBA Finals (OKC vs. IND):** * **Model Prediction:** OKC ($68.0\%$ Win Probability / $-312$ Fair Market Odds)
   * **Real-World Outcome:** OKC won the championship 4-3. The model successfully isolated the structural advantages and elite talent concentration (`Star_Power`) of the Thunder.

2. **Eastern Conference Finals (IND vs. NYK):**
   * **Model Prediction:** NYK ($58.5\%$ Win Probability)
   * **Real-World Outcome:** IND won the series 4-2. 
   * **Post-Mortem Analysis:** This "miss" highlights a classic machine learning limitation: *Contextual Variance*. While NYK possessed superior season-long statistical aggregates, real-world postseason injuries severely depleted their active rotation. Future iterations of this model will incorporate a **weighted active-roster availability index** to account for real-time player injuries.

### The $+EV$ (Wagering Value) Framework
By converting our model's probabilities into **Fair Market Odds** (e.g., $-312$ for OKC), we can actively scan commercial sportsbook lines (DraftKings, FanDuel) for market inefficiencies. If a sportsbook offers OKC at $-180$ (implying a $64.3\%$ probability) while our model evaluates them at $68.0\%$, this creates a mathematically positive expected value ($+EV$) wager opportunity.